In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Load the CSV file
df = pd.read_csv("Mess_Quarter_Raw.csv")

# Show first 5 rows before cleaning
df.head()


,date,roll_no,lunch,dinner,sweet,special_diet,dish_name,price,meal_type,quantity
0,2025-07-01,HM029,No,NaN,No,Vegan,Mix Veg,55.0,Lunch,1
1,2025-07-01,HM007,Yes,Yes,No,Vegan,Mix Veg,45.0,Lunch,1
2,2025-07-01,HM071,No,NaN,No,Gluten-Free,Chole Rice,45.0,Lunch,2
3,2025-07-01,HM063,Yes,NaN,No,Diabetic,Dal Tadka,65.0,Both,3
4,2025-07-01,HM058,NaN,Yes,Yes,Diabetic,Dal Tadka,45.0,Both,1


In [ ]:
# Show number of rows and columns before cleaning
print("Shape Before Cleaning:", df.shape)

Shape Before Cleaning: (625, 10)


In [ ]:
# Check missing values before cleaning
df.isnull().sum()


,0
date,0
roll_no,0
lunch,214
dinner,223
sweet,227
special_diet,245
dish_name,0
price,84
meal_type,0
quantity,0


In [ ]:
# Count duplicate rows before cleaning
df.duplicated().sum()


np.int64(5)

In [ ]:
# Show data types of each column
df.dtypes

,0
date,object
roll_no,object
lunch,object
dinner,object
sweet,object
special_diet,object
dish_name,object
price,float64
meal_type,object
quantity,int64


In [ ]:
# Convert column names to lowercase and remove spaces
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")
df.columns

Index(['date', 'roll_no', 'lunch', 'dinner', 'sweet', 'special_diet',
       'dish_name', 'price', 'meal_type', 'quantity'],
      dtype='object')

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()


In [ ]:
# Convert date column to datetime format
df["date"] = pd.to_datetime(df["date"], errors="coerce")


In [ ]:
# Remove rows where date is missing
df = df.dropna(subset=["date"])


In [ ]:
# Replace missing meal values with 'No'
df[["lunch", "dinner", "sweet"]] = df[["lunch", "dinner", "sweet"]].fillna("No")


In [ ]:
# Replace missing price with median value
df["price"] = df["price"].fillna(df["price"].median())


In [ ]:
# Replace zero or negative price with median
df.loc[df["price"] <= 0, "price"] = df["price"].median()

In [ ]:
# Replace missing quantity with 1
df["quantity"] = df["quantity"].fillna(1)

In [ ]:
# Make Yes/No values consistent
df["lunch"] = df["lunch"].str.capitalize()
df["dinner"] = df["dinner"].str.capitalize()
df["sweet"] = df["sweet"].str.capitalize()


In [ ]:
# Count number of meals taken
df["meal_count"] = (
    (df["lunch"] == "Yes").astype(int) +
    (df["dinner"] == "Yes").astype(int) +
    (df["sweet"] == "Yes").astype(int)
)


In [ ]:
# Remove rows where no meals were taken
df = df[df["meal_count"] > 0]


In [ ]:
# Calculate total bill
df["total_bill"] = df["price"] * df["quantity"] * df["meal_count"]


In [ ]:
# Extract month name from date
df["month"] = df["date"].dt.month_name()



In [ ]:
# Categorize price into Low, Medium and High
df["cost_category"] = pd.cut(
    df["price"],
    bins=[0,50,100,df["price"].max()],
    labels=["Low","Medium","High"]
)


In [ ]:
# Show number of rows and columns after cleaning
print("Shape After Cleaning:", df.shape)


Shape After Cleaning: (425, 14)


In [ ]:
# Check missing values after cleaning
df.isnull().sum()


,0
date,0
roll_no,0
lunch,0
dinner,0
sweet,0
special_diet,174
dish_name,0
price,0
meal_type,0
quantity,0


In [ ]:
# Check duplicate rows after cleaning
df.duplicated().sum()


np.int64(0)

In [ ]:
# Calculate total revenue
print("Total Revenue:", df["total_bill"].sum())

# Revenue by month
df.groupby("month")["total_bill"].sum()

Total Revenue: 105845.0


,total_bill
month,
August,36045.0
July,33575.0
September,36225.0


In [ ]:
# Export final cleaned data for SQL use
df.to_csv("Final_SQL_Ready_Data.csv", index=False)
